# **03 - Neon connection and raw load of all Berka tables**

## GOAL
- Test the Python connection to the Neon database.
- First step of the pipeline: if Python can't reach the database, nothing else will work.
- The 8 Berka tables have been uploaded to the raw schema. The `trans` table has more than a million rows, so it was uploaded in batches. The rows of each table were verified by comparing them with the CSV files.
- The next step is to move this code to `src/load.py`, so GitHub Actions can run it without the notebook.

Import the libraries used in this notebook.

In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd
import os
import sys
sys.path.append("..")
from src.extract import load_table


.env keeps the credential out of the code and GitHub. GitHub Actions doesn't load .env automatically like VS code does, so we call load_dotenv explicitly.

In [2]:
load_dotenv("../.env")

True

os.getenv reads the `DATABASE_URL` stored in .env (the Neon access key). Never print db_url because it would show the password, so we only check it with `is not None`. 

In [3]:
db_url = os.getenv("DATABASE_URL")

In [4]:
db_url is not None 

True

The engine variable holds the connection setup. We use the key and pool_pre_ping, because Neon goes to sleep after about 5 idle minutes and the ping avoids using a stale connection. 

SQLAlchemy 2.1 uses psycopg 3 by default, so we switched from psycopg 2 to psycopg 3.

In [5]:
engine = create_engine(db_url, pool_pre_ping=True) 

Minimal connection test. If it returns 1, Neon answered.  

In [6]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1")).scalar()
    print(result)

1


Here we use `engine.begin()` because it saves the change automatically. With `connect()` we could do it too, but we would have to write `conn.commit()` at the end for the change to take place.
We create a schema called raw to store all the raw data. Later we will create processed to store the clean data.
We use `IF NOT EXISTS` so we can re-run the cell without errors.

In [7]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

Here we load the loan table into Neon with `to_sql` using the following parameters:
* **loan:** The table's name
* **engine:** The Neon access setup 
* **schema:** the raw schema
* **if_exists:** If the table already exists, we can re-run without duplicating it.
* **index:** avoids loading a column with index numeration.

psycopg 3 doesn't report how many rows were loaded (it returns -1), so we verify it in the next cell.

In [8]:
load_table("loan").to_sql("loan", engine, schema="raw", if_exists="replace", index=False)

-1

With COUNT we verify the 682 rows in the loan table. No error is not enough, so we check that all the rows arrived.

In [9]:
pd.read_sql("SELECT COUNT(*) FROM raw.loan", engine)

,count
0,682


I made a list called tables to upload the tables, and I wrote the table names in it.

Then I iterated over the list with a variable called table_name and uploaded each table to Neon with `to_sql`, using the same parameters as before. For each table, a print shows that it was loaded.


In [10]:
tables = ["account", "card", "client", "disp", "district", "order"]
for table_name in tables:
    load_table(table_name).to_sql(table_name, engine, schema="raw", if_exists="replace", index=False)
    print(f"{table_name} loaded")

account loaded
card loaded
client loaded
disp loaded
district loaded
order loaded


This cell checks if the number of rows in each CSV files matches with the database.

For this reason, I iterated over the tables with `table_name`, counted the rows in `csv_rows` and `db_rows` and printed the results with `ok` to show if they match.

In [11]:
for table_name in tables:
    csv_rows = load_table(table_name).shape[0]
    db_rows = pd.read_sql(f'SELECT COUNT(*) FROM raw."{table_name}"', engine).iloc[0, 0]
    print(f"{table_name}, csv={csv_rows}, db_rows={db_rows}, ok={csv_rows == db_rows}")

account, csv=4500, db_rows=4500, ok=True
card, csv=892, db_rows=892, ok=True
client, csv=5369, db_rows=5369, ok=True
disp, csv=5369, db_rows=5369, ok=True
district, csv=77, db_rows=77, ok=True
order, csv=6471, db_rows=6471, ok=True


The table `trans` was uploaded. This time I used two new parameters:
- **{"bank": str}:** the `bank` column is read as text (str).
- **chunksize:** It loads the data in batches of 10,000 rows.

I loaded the `trans` table separately because it is large: a little more than a million rows. With chunksize, it is loaded in batches of 10,000 rows, which uses less memory. The result is -106 because there were 106 batches and each one returns -1.

I used dtype because the `bank` column has many NaN values and text values. Pandas reads NaN as a float, so the column had mixed types and showed a `DtypeWarning`. After checking the data, I saw that it is text, so I read it as str.

In [ ]:
load_table("trans", {"bank": str}).to_sql("trans", engine, schema="raw", if_exists="replace", index=False, chunksize=10000)

-106

I counted the rows and checked that the Neon database matches the CSV file.

In [ ]:
csv_rows = load_table("trans",{"bank": str}).shape[0]
db_rows = pd.read_sql("SELECT COUNT(*) FROM raw.trans", engine).iloc[0, 0]
print(f"trans: csv={csv_rows}, db_rows={db_rows}, ok={csv_rows == db_rows}")

trans: csv=1056320, db_rows=1056320, ok=True
